In [1]:
# --- 1. IMPORTS ---
import os
import gc
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Imports locais (Assumindo que seus arquivos .py estão na pasta cnn_utils)
try:
    import utils.processamento_dados as proc_dados
    import utils.metricas_e_visualizacao as met_vil
except ImportError:
    print("AVISO: Módulos 'cnn_utils' não encontrados. Certifique-se de que estão no caminho.")


2026-01-09 07:43:26.103716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-09 07:43:26.630883: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-09 07:43:26.725622: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-09 07:43:27.401812: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# --- 2. CONFIGURAÇÕES DE AMBIENTE E CAMINHOS ---
# Configuração básica de GPU (Memory Growth) para evitar alocação total imediata
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

I0000 00:00:1767955513.879615  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955513.882254  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955513.882310  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.


In [3]:
# DEFINA SEUS CAMINHOS AQUI:
BASE_DIR = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI"
data_3t_dir = os.path.join(BASE_DIR, "ADNI_3_4_NORMALIZED")
results_dir = os.path.join(BASE_DIR, "results", "fine_tuning_3t_k_fold")
os.makedirs(results_dir, exist_ok=True)

# Nome do modelo pré-treinado (Treinado no dataset 1.5T)
# Exemplo: "binary_classifier_noise_200_epochs_batch_16_2_classes.keras"
pretrained_model_path = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/results_fit_15_predict_3/test_1/binary_classifier_noise_200_epochs_batch_15_2_classes.keras"

# --- 3. DEFINIÇÃO DO MODELO (NOVA ARQUITETURA) ---
def create_model_3d(input_shape, n_classes):
    """
    Cria a arquitetura leve com LeakyReLU e Dropout, adaptável para n_classes.
    """
    inputs = Input(shape=input_shape)  # (D, H, W, C)

    # === Camada 1 ===
    x = layers.Conv3D(4, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Camada 2 ===
    x = layers.Conv3D(8, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Camada 3 ===
    x = layers.Conv3D(16, (3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)
    x = layers.AveragePooling3D(pool_size=(3, 3, 3), padding='same')(x)
    x = layers.Dropout(0.3)(x)

    # === Classificador ===
    x = layers.Flatten()(x)

    x = layers.Dense(16, kernel_regularizer=l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LeakyReLU(negative_slope=0.3)(x)

    # === Saída ===
    if n_classes == 2:
        outputs = layers.Dense(1, activation='sigmoid')(x)
    else:
        outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="Small_Leaky_3D_Model")
    return model


In [ ]:
# === 4. PREPARAÇÃO DOS DADOS 3T (União de todas as pastas) ===
print("--- Carregando e Unificando Dados 3T ---")

dir_3t_train = os.path.join(data_3t_dir, "train")
dir_3t_val = os.path.join(data_3t_dir, "validation")
dir_3t_test = os.path.join(data_3t_dir, "test")

parts_X = []
parts_y = []

# Carrega tudo usando a função do seu módulo local
for d in [dir_3t_train, dir_3t_val, dir_3t_test]:
    print(f"Lendo pasta: {d}")
    try:
        x_part, y_part = proc_dados.load_nifti_data(d)
        if len(x_part) > 0:
            parts_X.append(x_part)
            parts_y.append(y_part)
    except Exception as e:
        print(f"Erro ao ler pasta {d}: {e}")

if len(parts_X) > 0:
    X_3t_all = np.concatenate(parts_X, axis=0)
    y_3t_all = np.concatenate(parts_y, axis=0)
    print(f"Total de dados 3T para K-Fold: {X_3t_all.shape}")
else:
    raise ValueError("Não foram encontrados dados nas pastas 3T.")

# === 5. CONFIGURAÇÃO DO K-FOLD E MODELO ===
K_FOLDS = 5
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

# Listas para armazenar métricas de cada fold
fold_accuracies = []
fold_aucs = []

all_true_labels = []
all_pred_labels = []

print(f"\n>>> INICIANDO FINE-TUNING (K-FOLD = {K_FOLDS}) <<<")
print(f"Modelo Base para carregar pesos: {pretrained_model_path}")
print("Estratégia: Congelar 1º Bloco Convolucional\n")

# Verifica se o modelo base existe antes de começar o loop
if not os.path.exists(pretrained_model_path):
    raise FileNotFoundError(f"Modelo pré-treinado não encontrado em: {pretrained_model_path}")

--- Carregando e Unificando Dados 3T ---
Lendo pasta: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/train
Carregando dados de: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/train


Lendo ad: 100%|██████████| 88/88 [00:10<00:00,  8.01it/s]


Dados carregados. Shape: (699, 156, 195, 160, 1)
Lendo pasta: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/validation
Carregando dados de: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/validation


Lendo ad: 100%|██████████| 27/27 [00:03<00:00,  6.76it/s]


Dados carregados. Shape: (202, 156, 195, 160, 1)
Lendo pasta: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/test
Carregando dados de: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_3_4_NORMALIZED/test


Lendo ad: 100%|██████████| 13/13 [00:01<00:00,  7.20it/s]


Dados carregados. Shape: (102, 156, 195, 160, 1)
Total de dados 3T para K-Fold: (1003, 156, 195, 160, 1)

>>> INICIANDO FINE-TUNING (K-FOLD = 5) <<<
Modelo Base para carregar pesos: /mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/results_fit_15_predict_3/test_1/binary_classifier_noise_200_epochs_batch_15_2_classes.keras
Estratégia: Congelar 1º Bloco Convolucional



: 

In [ ]:
# === 6. LOOP DE TREINAMENTO ===
for fold, (train_idx, val_idx) in enumerate(skf.split(X_3t_all, y_3t_all)):
    print(f"\n--- Fold {fold+1}/{K_FOLDS} ---")
    
    # Separação dos dados para este fold
    X_train_fold, X_val_fold = X_3t_all[train_idx], X_3t_all[val_idx]
    y_train_fold, y_val_fold = y_3t_all[train_idx], y_3t_all[val_idx]
    
    # 1. Instanciar arquitetura limpa
    # Usa o shape do dado carregado para definir a entrada
    model_ft = create_model_3d(X_train_fold[0].shape, 2)
    
    # 2. Carregar pesos do pré-treino (Transfer Learning)
    # Isso inicializa o modelo com o que ele aprendeu no 1.5T
    print("   Carregando pesos...")
    try:
        model_ft.load_weights(pretrained_model_path) # by_name=True pode ajudar se houver incompatibilidade leve
    except Exception as e:
        print(f"ERRO CRÍTICO ao carregar pesos: {e}")
        break
    
    # 3. Congelar o Primeiro Bloco
    # layer[0]=Input, layer[1]=Conv1, layer[2]=BN, layer[3]=Leaky, layer[4]=Pool, layer[5]=Drop
    print("   Congelando camadas iniciais (0 a 5)...")
    for i in range(6):
        model_ft.layers[i].trainable = False
        
    # 4. Compilar (Necessário re-compilar após alterar trainable)
    # Usamos LR menor (1e-4) para fine-tuning
    model_ft.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    # 5. Callbacks
    es_ft = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
    
    # 6. Treinar
    history_ft = model_ft.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=50, 
        batch_size=8, 
        callbacks=[es_ft],
        verbose=1
    )
    
    # 7. Avaliar
    scores = model_ft.evaluate(X_val_fold, y_val_fold, verbose=0)
    acc = scores[1]
    auc = scores[2]
    
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    
    print(f"   Resultado Fold {fold+1}: Acurácia = {acc:.4f} | AUC = {auc:.4f}")
    
    y_pred_prob = model_ft.predict(X_val_fold, verbose=0)
        
    # Converte para classes binárias (Threshold 0.5)
    # Se sua saída for sigmoid (1 neurônio), usa > 0.5
    y_pred_bin = (y_pred_prob > 0.5).astype('int32').flatten()
    
    # Acumula nas listas globais
    all_true_labels.extend(y_val_fold)
    all_pred_labels.extend(y_pred_bin)

    # Limpeza de memória
    tf.keras.backend.clear_session()
    del model_ft
    gc.collect()


--- Fold 1/5 ---


I0000 00:00:1767955983.008072  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955983.017880  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955983.017928  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955984.121108  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1767955984.125943  843760 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:21:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-09

   Carregando pesos...
   Congelando camadas iniciais (0 a 5)...


In [ ]:
# === 7. RESULTADOS FINAIS ===
print(f"\n==========================================")
print(f"MÉDIA FINAL ({K_FOLDS} Folds) - Fine Tuning 3T")
print(f"Acurácia: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"AUC:      {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")
print(f"==========================================")

print("\nGerando Matriz de Confusão Agregada...")

# Calcula a matriz usando scikit-learn
cm = confusion_matrix(all_true_labels, all_pred_labels)

# Define nomes das classes (ajuste conforme seu caso, ex: ['CN', 'AD'])
# Se y=0 é CN e y=1 é AD:
class_names = ['CN', 'AD'] 

# Plota
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')

ax.set_title(f"Matriz de Confusão Agregada (Total {len(all_true_labels)} amostras)")

# Salva em arquivo
save_path = os.path.join(results_dir, f"aggregated_confusion_matrix_{K_FOLDS}folds.png")
plt.savefig(save_path, dpi=300)
print(f"Matriz de confusão salva em: {save_path}")


plt.show()